#### Initialize Context & Seed Ledger

In [2]:
import sys
from pathlib import Path
from datetime import datetime
HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path: sys.path.insert(0, str(PARENT))

from server.scripts.h3.h3_load import load_geodf_from_csv
SEED_LEDGER = Path("../../map/seed_ledger_level_1.csv")
if not SEED_LEDGER.parent.exists():
    raise FileNotFoundError(f"SEED_LEDGER Not Found.")
else:
    display("Loading existing seed ledger")
    seed_geodf = load_geodf_from_csv(SEED_LEDGER)
    display(seed_geodf.head(2))

'Loading existing seed ledger'

,tile_id,seed_index,tile_path_id,h3_res,level,center_lat,center_lon,tile_size_m,geometry,children,checked,fetch_success,current
0,88194ad305fffff,0,0,8,0,51.514783,-0.090079,531.415,"POLYGON ((-0.08885 51.51926, -0.09556 51.51807...",0,True,True,False
1,88194ad32bfffff,1,1,8,0,51.509118,-0.098012,531.415,"POLYGON ((-0.09679 51.51359, -0.10349 51.5124,...",0,True,True,False


#### Select Cells

In [3]:
SELECTED_CELLS = []
COORDINATES = [51.461127, -0.008581, 1200]
ID_FIELD = "tile_id"
from server.scripts.h3.h3_selecttiles import select_tiles_in_radius
from server.scripts.h3.h3_visualizemap import visualize_seed

# if not SELECTED_CELLS:
#     seed_select = select_tiles_in_radius(seed_geodf, COORDINATES[0], COORDINATES[1], COORDINATES[2])
#     SELECTED_CELLS = seed_select[ID_FIELD].tolist()

existing_cells = seed_geodf[(seed_geodf['checked']==True) & (seed_geodf['fetch_success']==True)][ID_FIELD].tolist()
SELECTED_CELLS = [CELL for CELL in SELECTED_CELLS if CELL not in existing_cells]

seed_geodf["current"] = False
seed_geodf["current"] = seed_geodf[ID_FIELD].apply(lambda x: x in SELECTED_CELLS)
print(f"Selected {seed_geodf['current'].sum()} cells for division.")

Selected 0 cells for division.


In [4]:
from server.scripts.h3.h3_load import load_boundary_from_json

print("Updating Ledger")
seed_geodf.to_csv(SEED_LEDGER, index=False)
print("Updating Map")
inner_union = load_boundary_from_json(json_path='../get_seed_map_level1/inner_london_boundary.json')
visualize_seed(seed_geodf, inner_union, output_path=SEED_LEDGER.with_suffix(".html"))

Updating Ledger
Updating Map


Saved seed H3 map to: ..\..\map\seed_ledger_level_1.html


#### Run APIs

In [ ]:
ENABLE_API = True
from server.scripts.h3.h3_subdivision import run_h3_recursive_division

seed_select = seed_geodf[seed_geodf['current'] == True]
res_geodf = await run_h3_recursive_division(seed_select, inner_union, disable_api=not ENABLE_API)

API calls executed for 341: 1 | failures: 0
Run complete | Final Cell Count: 1 | Stats: {'api_calls': 1, 'discarded': 0, 'added': 0, 'api_failures': 0}


In [ ]:
cell_geodf = res_geodf.copy()
cell_geodf["tile_id"] = res_geodf["tile_id"].astype(str)
cell_geodf["seed_index"] = res_geodf["seed_index"].astype(int)
cell_geodf["tile_path_id"] = res_geodf["tile_path_id"].astype(str)

#### File API Response and Update Ledger

In [ ]:
cell_success = cell_geodf[cell_geodf['fetch_success'] == True]
if ENABLE_API and not cell_success.empty:
    print("Fetch Complete. Updating Ledger.")
    success_ids = cell_success['tile_id'].dropna()
    seed_geodf.loc[seed_geodf['tile_id'].isin(success_ids), 'fetch_success'] = True
    seed_geodf.loc[(seed_geodf['current'] == True) & 
        (seed_geodf['fetch_success'] == True), 'checked'] = True
    seed_geodf['current'] = False
    seed_geodf.to_csv(SEED_LEDGER, index=False)
    visualize_seed(seed_geodf, inner_union, output_path=SEED_LEDGER.with_suffix(".html"))
else:
    print("Response InValid")

Fetch Complete. Updating Ledger.
Saved seed H3 map to: ..\..\map\seed_ledger_level_1.html


In [ ]:
import pandas as pd
import geopandas as gpd
from server.scripts.h3.h3_visualizemap import visualize_divisions
OUT_LEDGER = SEED_LEDGER.parent / "seed_ledger_level_2.csv"
BATCH_LEDGER = SEED_LEDGER.parent / "batch" / f"{datetime.now().strftime('%Y-%m-%d')}-{len(seed_select)}-{len(cell_geodf)}.csv"
if not BATCH_LEDGER.parent.exists(): 
    BATCH_LEDGER.parent.mkdir(parents=True, exist_ok=True)

cell_success = cell_geodf[cell_geodf['fetch_success'] == True]

if ENABLE_API and not cell_success.empty:
    if not OUT_LEDGER.exists():
        print("Fetch Complete. Creating New Division Ledger.")
        new_geodf = cell_success.copy()
        new_geodf.to_csv(OUT_LEDGER, index=False)
    else:
        print("Fetch Complete. Updating Existing Division Ledger.")
        existing_geodf = load_geodf_from_csv(OUT_LEDGER)
        merged = pd.concat([existing_geodf, cell_success], ignore_index=True)
        merged["tile_id"] = merged["tile_id"].astype(str)
        merged = merged.drop_duplicates(
            subset=["tile_id"], 
            keep="last"
        ).sort_values(by="seed_index")
        new_geodf = gpd.GeoDataFrame(merged, geometry="geometry", crs=existing_geodf.crs)
        new_geodf.to_csv(OUT_LEDGER, index=False)

    cell_success.to_csv(BATCH_LEDGER, index=False)
    visualize_divisions(new_geodf, inner_union, output_path=OUT_LEDGER.with_suffix(".html"))

else:
    print("Response InValid")

Fetch Complete. Updating Existing Division Ledger.
Saved mock adaptive H3 map to: ..\..\map\seed_ledger_level_2.html


#### Update Division Map

In [ ]:
# OUT_LEDGER = SEED_LEDGER.parent / "seed_ledger_level_2.csv"
# from server.scripts.h3.h3_visualizemap import visualize_divisions
# from server.scripts.h3.h3_load import load_boundary_from_json
# inner_union = load_boundary_from_json(json_path='../get_seed_map_level1/inner_london_boundary.json')

division_geodf = load_geodf_from_csv(OUT_LEDGER)
division_geodf = division_geodf[division_geodf['children']==0]
visualize_divisions(division_geodf, inner_union, output_path=OUT_LEDGER.with_suffix(".html"))

Saved mock adaptive H3 map to: ..\..\map\seed_ledger_level_2.html
